# Multi-Domain RAG Agents (5 domains, Colab edition)

Self-contained version of **AIProject_RAG_agents**: LangGraph multi-agent pipeline + hybrid RAG
(MiniLM vectors + BM25) over **5 RAGBench domains** (biomedical, legal, finance, customer support,
general knowledge), answered by a **Groq** LLM. Triage picks the domain; retrieval searches only that domain.

**v2.5:** finance answers get each retrieved report's table; error codes in a question get extra keyword matches; the safety gate removes an unverifiable citation instead of discarding the whole answer.

**v2.6:** finance tables are shown to the model with each value labelled by its column, so it reads the right column (for example *total* rather than *after 2020*).

**Flow:** `triage (+ domain) → hybrid_retrieval (domain-filtered) → drug_interaction_check → evidence_synthesis → safety_gate`

### How to run
1. Add your Groq key: click the 🔑 **Secrets** icon in the left sidebar → **Add new secret** →
   name `GROQ_API_KEY`, value `gsk_...` → turn on **Notebook access**.
   (If you skip this, cell 2 will prompt you for the key.)
2. Recommended: **Runtime → Change runtime type → T4 GPU** (embedding ~4,800 chunks is slow on CPU).
3. **Runtime → Run all**. The first run downloads 5 sub-datasets and the embedding model (a few minutes).
4. Use the chat UI in the last cell, or call `ask("...")` directly.

> Research prototype only — not for diagnosis, treatment, medication changes, or dosing.

## 1. Install packages

In [ ]:
!pip install -q "datasets>=3.0,<5.0" "langchain>=1.0,<2.0" "langchain-core>=1.0,<2.0" \
    "langchain-groq>=1.0,<2.0" "langchain-huggingface>=1.0,<2.0" "langgraph>=1.0,<2.0" \
    "rank-bm25>=0.2.2,<0.3" "requests>=2.32,<3.0" "sentence-transformers>=3.0,<6.0" \
    langchain-chroma gradio

## 2. Settings and Groq API key

- `DOMAINS`: which RAGBench sub-datasets to index and how many train rows of each (`0` = all rows).
  `cuad` rows are whole contracts (~60,000 characters each), so 200 cuad rows is the largest share of the index (about 10,000 chunks); use a GPU runtime.
- **Runtime: use a GPU** (Runtime → Change runtime type → T4 GPU). Embedding the ~14,700 chunks takes
  ~2–3 minutes on a T4 GPU but ~1–1.5 hours on Colab's CPU.
- `QUICK_TEST = True` indexes 50 rows per domain (~3,700 chunks) for a fast demo run.
- `PERSIST_DIR`: `None` keeps the vector index in memory (rebuilt every run). Set a folder path to save it
  with **Chroma** so later runs skip re-embedding — use a Google Drive path (run cell 3) to keep it across sessions.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    pass  # not in Colab, or the secret is missing / access not granted
if not os.getenv("GROQ_API_KEY"):
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Groq API key (gsk_...): ")

GROQ_MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
import torch
EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# sub-dataset -> domain label + train rows to index (0 = all rows)
DOMAINS = {
    "covidqa":  {"domain": "biomedical",       "max_rows": 200},
    "cuad":     {"domain": "legal",            "max_rows": 200},
    "finqa":    {"domain": "finance",          "max_rows": 200},
    "techqa":   {"domain": "customer_support", "max_rows": 200},
    "hotpotqa": {"domain": "general",          "max_rows": 200},
}
QUICK_TEST = False   # True = 50 rows per domain for a fast demo run
if QUICK_TEST:
    for spec in DOMAINS.values():
        spec["max_rows"] = min(spec["max_rows"] or 50, 50)

TOP_K = 8             # evidence chunks per answer (v2.4: 5 -> 8 found the answer passage more often)
GLOBAL_BACKSTOP = 2   # of the TOP_K chunks, 2 come from all domains in case triage misroutes
REPORT_TABLES = 4     # finance: up to 4 table chunks of the retrieved reports are added (v2.5)
CODE_EXTRA = 2        # extra chunks matching error codes / identifiers in the question (v2.5)
PERSIST_DIR = None                # e.g. "/content/drive/MyDrive/rag_vector_db"

print("Groq key loaded:", os.environ["GROQ_API_KEY"].startswith("gsk_"))
print("Embedding device:", EMBEDDING_DEVICE)
print("Rows per domain:", {name: spec["max_rows"] for name, spec in DOMAINS.items()})
if EMBEDDING_DEVICE == "cpu":
    print(
        "\n⚠️  No GPU: embedding ~14,700 chunks on CPU takes about 1–1.5 hours.\n"
        "   Switch to Runtime → Change runtime type → T4 GPU (about 2–3 minutes), "
        "or set QUICK_TEST = True."
    )

## 3. (Optional) Mount Google Drive — only if `PERSIST_DIR` points to Drive

In [ ]:
if PERSIST_DIR and PERSIST_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipped: PERSIST_DIR is not on Google Drive.")

## 4. Imports, constants and data models

In [ ]:
import json
import re
from collections import defaultdict
from typing import Literal, TypedDict

import requests
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi

DATASET_NAME = "galileo-ai/ragbench"
DOMAIN_VOTE_MIN = 3  # of the global top-5 chunks needed to override the triage domain

DOMAIN_RULES = {
    "biomedical": "Do not diagnose, prescribe, recommend starting/stopping/changing medicine, or give a patient-specific dose.",
    "legal": "Do not give legal advice or say what anyone should sign or do; describe only what the contract excerpts say.",
    "finance": "Do not give investment, tax or trading advice; show any calculation step by step using only numbers in the excerpts.",
    "customer_support": "Give only steps, commands, settings and versions that appear in the excerpts.",
    "general": "Answer only from the excerpts.",
    "unknown": "Answer only from the excerpts.",
}
DATASET_URL = "https://huggingface.co/datasets/galileo-ai/ragbench"
DATASET_LICENSE = "CC BY 4.0"
OPENFDA_LABEL_URL = "https://api.fda.gov/drug/label.json"
MAX_MEDICATIONS = 6
MAX_LABEL_CHARS = 3_000

DISCLAIMER = (
    "Research prototype only — not medical, legal, financial or technical advice. "
    "RAGBench is benchmark data, not current professional guidance. Verify all "
    "information with a qualified expert. Do not enter personal or confidential "
    "information; prompts are sent to the "
    "configured Groq model provider."
)
EMERGENCY_PATTERNS = (
    r"\b(chest pain|pressure in (my|the) chest)\b",
    r"\b(can't breathe|cannot breathe|difficulty breathing|severe shortness of breath)\b",
    r"\b(severe allergic reaction|anaphylaxis|throat is closing)\b",
    r"\b(uncontrolled bleeding|severe bleeding|bleeding won't stop)\b",
    r"\b(face|arm|leg) weakness\b",
    r"\b(sudden confusion|loss of consciousness|unconscious)\b",
)


class TriageResult(BaseModel):
    domain: Literal["biomedical", "legal", "finance", "customer_support", "general", "unknown"] = Field(
        description=(
            "Knowledge domain. biomedical = COVID-19, viruses, infections, immunology, clinical or "
            "biomedical research studies (including 'this study' questions). legal = clauses or terms "
            "of a contract or agreement (termination, licence, IP, liability, renewal, escrow). "
            "finance = numbers from company annual reports: revenue, expenses, rent, amortization, "
            "shares, returns, percentages, ratios, differences between years, amounts in millions. "
            "customer_support = IBM software products, errors, fixes, configuration, security "
            "bulletins. general = Wikipedia-style facts about people, places, films, bands, sport. "
            "unknown only if none fit."
        )
    )
    document_name: str = Field(
        default="",
        description=(
            "Name of the specific contract or agreement, or of its parties/companies, if the "
            "question names one (e.g. 'Moelis strategic alliance agreement'); empty otherwise."
        ),
    )
    category: str = Field(description="Short category for the question.")
    severity: Literal["low", "moderate", "high", "critical"] = Field(
        description="Potential clinical risk; use critical for a possible emergency."
    )
    emergency_concern: bool = Field(
        description="True if the user describes a possible immediate emergency."
    )
    clinician_review: bool = Field(
        description="True if the question needs individualized clinician review."
    )
    medications: list[str] = Field(
        default_factory=list,
        description="Medication names explicitly mentioned by the user, if any.",
    )
    rationale: str = Field(description="Brief explanation without diagnosing.")


class InteractionFinding(BaseModel):
    medications: list[str] = Field(
        description="The medication names involved in the possible interaction."
    )
    source_id: str = Field(description="An FDA source ID provided in the input.")
    exact_quote: str = Field(
        description="An exact excerpt from that label supporting the concern."
    )


class InteractionAssessment(BaseModel):
    findings: list[InteractionFinding] = Field(default_factory=list)
    summary: str = Field(
        description="Brief screening result; do not claim a medication combination is safe."
    )
    clinician_review: bool = Field(
        description="True when an interaction concern or uncertainty warrants review."
    )


class SafetyAssessment(BaseModel):
    unsafe_or_unsupported_claims: list[str] = Field(
        default_factory=list,
        description=(
            "Claims in the draft that are unsupported by the context, or that "
            "give a diagnosis, dose, or individualized treatment instruction."
        ),
    )
    requires_clinician_review: bool = Field(
        description="True for uncertainty or individualized clinical decisions."
    )


class DrugLabel(TypedDict):
    source_id: str
    medication: str
    status: str
    url: str
    text: str


class ClinicalState(TypedDict, total=False):
    question: str
    user_medications: list[str]
    document: str  # optional contract doc_key chosen by the user
    triage: TriageResult
    medications: list[str]
    evidence: list[Document]
    drug_labels: list[DrugLabel]
    interaction_assessment: InteractionAssessment
    draft: str
    answer: str
    review_required: bool
    safety_notes: list[str]

## 5. Load the 5 RAGBench domains and chunk the passages

In [ ]:
def _tokens(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)*", text.lower())


def _chunk_text(text: str, size: int = 1_200, overlap: int = 150) -> list[str]:
    text = text.strip()
    if not text:
        return []

    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start + size // 2, end)
            if boundary > start:
                end = boundary
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


_TITLE_NOISE = re.compile(
    r"^(?:\d+\s+|table of contents\s*|execution (?:version|copy)\s*|exhibit\s+[\w.()-]+\s*"
    r"|confidential treatment requested.*?\*\*\*\W*\s*"
    r"|this exhibit has been redacted.*?commission\.\s*)",
    re.IGNORECASE,
)
CONTRACTS: dict[str, str] = {}  # doc_key -> contract title (legal domain)
CONTRACT_HEADS: dict[str, str] = {}  # doc_key -> opening text, for matching party names
ROW_DOCS: defaultdict[str, list[str]] = defaultdict(list)  # 'config:row' -> doc_keys of its documents
TABLES: dict[str, list[list[str]]] = {}  # doc_key -> table rows (FinQA tables are JSON lists)


def _parse_table(text: str) -> list[list[str]] | None:
    if not text.lstrip().startswith("[["):
        return None
    try:
        rows = json.loads(text)
    except ValueError:
        return None
    if isinstance(rows, list) and len(rows) > 1 and all(isinstance(row, list) for row in rows):
        return [[str(cell) for cell in row] for row in rows]
    return None


def _table_text(rows: list[list[str]]) -> str:
    """'[["", "total", "2016"], ["debt", "9878", "1125"]]' -> 'debt: total = 9878; 2016 = 1125'."""
    header = rows[0]
    lines = ["Table (each value is labelled with its column):"]
    for row in rows[1:]:
        cells = [
            f"{(header[i] if i < len(header) and header[i].strip() else f'column {i}')} = {value}"
            for i, value in enumerate(row[1:], start=1)
        ]
        lines.append(f"- {row[0] or 'row'}: " + "; ".join(cells))
    return "\n".join(lines)


def evidence_texts(evidence: list[Document]) -> list[str]:
    """Text shown to the model for each chunk: a table chunk becomes its whole table, labelled."""
    texts, shown = [], {}
    for document in evidence:
        key = document.metadata.get("doc_key")
        if key in TABLES and key in shown:
            texts.append(f"(part of the table in [{shown[key]}])")
        elif key in TABLES:
            shown[key] = document.metadata["source_id"]
            texts.append(_table_text(TABLES[key]))
        else:
            texts.append(document.page_content)
    return texts


def _contract_head(text: str) -> str:
    head = " ".join(text[:900].split())
    while True:
        stripped = _TITLE_NOISE.sub("", head, count=1)
        if stripped == head:
            return head
        head = stripped


def _contract_title(text: str, limit: int = 110) -> str:
    head = _contract_head(text)
    return head if len(head) <= limit else head[:limit].rsplit(" ", 1)[0] + " …"


def _contract_name(row_id: str, text: str) -> str:
    """'MOELIS_CO_03_24_2014-EX-10.19-STRATEGIC ALLIANCE AGREEMENT__...' -> 'MOELIS — STRATEGIC ALLIANCE AGREEMENT'."""
    base = row_id.split("__")[0]
    if "__" not in row_id:
        return _contract_title(text)
    company = re.split(r"_|\s-\s", base)[0]
    kind = re.split(r"EX-[A-Za-z0-9.()]+[_-]", base)[-1] if "EX-" in base else base[len(company):]
    if kind == base:  # unusual id: keep the part after the last hyphen
        kind = base.rsplit("-", 1)[-1]
    kind = re.sub(r"^[\d_]+", "", kind).replace("_", " ")
    company = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", company).replace(",", ", ")  # GarrettMotionInc -> Garrett Motion Inc
    return f"{company.strip()} — {kind.strip(' _-')}"


def load_ragbench_documents() -> list[Document]:
    """Load source documents for every configured domain, never the benchmark's question/answer fields."""
    documents: list[Document] = []
    for config, spec in DOMAINS.items():
        dataset = load_dataset(DATASET_NAME, config, split="train")
        row_limit = min(len(dataset), spec["max_rows"]) if spec["max_rows"] else len(dataset)
        before, seen = len(documents), {}
        for row_number in range(row_limit):
            row = dataset[row_number]
            row_id = str(row.get("id") or row_number)
            source_documents = row.get("documents") or []
            for document_number, text in enumerate(source_documents, start=1):
                if not isinstance(text, str):
                    continue
                if text in seen:  # duplicate passage: indexed once, under the key it was first seen with
                    ROW_DOCS[f"{config}:{row_id}"].append(seen[text])
                    continue
                doc_key = f"{config}:{row_id}:{document_number}"
                seen[text] = doc_key
                ROW_DOCS[f"{config}:{row_id}"].append(doc_key)
                table = _parse_table(text) if spec["domain"] == "finance" else None
                if table:
                    TABLES[doc_key] = table
                title = ""
                if spec["domain"] == "legal":
                    title = _contract_name(row_id, text)
                    if title in CONTRACTS.values():
                        title += f" ({row_id.split('__')[0][-40:]})"  # keep names unique
                    CONTRACTS[doc_key] = title
                    CONTRACT_HEADS[doc_key] = f"{title} {_contract_head(text)}"
                for chunk_number, chunk in enumerate(_chunk_text(text), start=1):
                    documents.append(
                        Document(
                            page_content=chunk,
                            metadata={
                                "source_id": f"RAG-{len(documents) + 1:05d}",
                                "domain": spec["domain"],
                                "source": (
                                    f"RAGBench {config}, train row {row_id}, "
                                    f"document {document_number}, chunk {chunk_number}"
                                ),
                                "dataset_url": DATASET_URL,
                                "license": DATASET_LICENSE,
                                "doc_key": doc_key,
                                "title": title,
                            },
                        )
                    )
        print(f"{config:9} ({spec['domain']}): {row_limit} rows -> {len(documents) - before} chunks")
    print(f"{len(CONTRACTS)} contracts registered for contract-scoped legal search.")

    if not documents:
        raise ValueError(f"No source documents found in {DATASET_NAME!r} for {list(DOMAINS)}.")
    return documents

## 6. Domain-filtered hybrid retriever (semantic + BM25, merged with Reciprocal Rank Fusion)

Uses Chroma on disk when `PERSIST_DIR` is set, otherwise an in-memory vector store.
Legal chunks also carry their contract (`doc_key`, `title`): when the question names a contract, or one is picked in the app, retrieval searches only that contract.
v2.5: `add_context` adds each retrieved finance report's table (FinQA stores a report's text and its table as separate documents) and the best keyword matches for error codes in the question.
Each chunk carries a `domain` tag; a search is limited to the domain triage picked (all domains when unknown).
The Chroma collection name includes the chunk count, so changing `DOMAINS` builds a fresh index
instead of mixing old and new chunk IDs.

In [ ]:
def _build_vector_store(documents: list[Document], embeddings: HuggingFaceEmbeddings):
    if not PERSIST_DIR:
        import time

        from tqdm.auto import tqdm

        store = InMemoryVectorStore(embeddings)
        started, batch = time.time(), 1_024
        for start in tqdm(range(0, len(documents), batch), desc=f"Embedding {len(documents)} chunks", unit="batch"):
            part = documents[start:start + batch]
            store.add_documents(part, ids=[d.metadata["source_id"] for d in part])
        print(f"Embedded {len(documents)} chunks in {(time.time() - started) / 60:.1f} min.")
        return store

    from langchain_chroma import Chroma

    store = Chroma(
        collection_name=f"ragbench_{len(documents)}_chunks",
        embedding_function=embeddings,
        persist_directory=PERSIST_DIR,
    )
    if store._collection.count() == 0:
        print(f"Embedding {len(documents)} chunks into Chroma at {PERSIST_DIR} (first run)...")
        from tqdm.auto import tqdm

        for start in tqdm(range(0, len(documents), 1_000), desc="Embedding into Chroma", unit="batch"):
            batch = documents[start:start + 1_000]
            store.add_documents(batch, ids=[d.metadata["source_id"] for d in batch])
    else:
        print(f"Loaded {store._collection.count()} chunks from Chroma at {PERSIST_DIR}.")
    return store


class HybridRetriever:
    """Combine semantic similarity and BM25 keyword rankings within one domain."""

    def __init__(self, documents: list[Document], embeddings: HuggingFaceEmbeddings):
        self.vector_store = _build_vector_store(documents, embeddings)
        self.documents_by_id = {
            document.metadata["source_id"]: document for document in documents
        }
        self.pools: defaultdict[str, list[Document]] = defaultdict(list)
        for document in documents:
            self.pools[document.metadata["domain"]].append(document)
        self.pools["all"] = documents
        self.embeddings = embeddings
        self.by_doc: defaultdict[str, list[Document]] = defaultdict(list)
        for document in documents:
            self.by_doc[document.metadata.get("doc_key", "")].append(document)
        self._doc_vectors: dict[str, list[list[float]]] = {}
        self.bm25 = {
            name: BM25Okapi([_tokens(document.page_content) for document in pool])
            for name, pool in self.pools.items()
        }

    def _ranked(self, query: str, key: str, limit: int) -> list[str]:
        pool = self.pools[key]
        limit = min(limit, len(pool))
        if key == "all":
            semantic_results = self.vector_store.similarity_search(query, k=limit)
        elif PERSIST_DIR:  # Chroma metadata filter
            semantic_results = self.vector_store.similarity_search(
                query, k=limit, filter={"domain": key}
            )
        else:  # InMemoryVectorStore takes a predicate
            semantic_results = self.vector_store.similarity_search(
                query, k=limit, filter=lambda document: document.metadata["domain"] == key
            )
        keyword_results = self.bm25[key].get_top_n(_tokens(query), pool, n=limit)

        scores: defaultdict[str, float] = defaultdict(float)
        for ranked_documents in (semantic_results, keyword_results):
            for rank, document in enumerate(ranked_documents, start=1):
                scores[document.metadata["source_id"]] += 1.0 / (60 + rank)
        return sorted(scores, key=scores.__getitem__, reverse=True)

    def search_document(self, query: str, doc_key: str, top_k: int = 5) -> list[Document]:
        """Hybrid search inside one contract (its chunks are embedded once, on first use)."""
        pool = self.by_doc.get(doc_key, [])
        if not pool:
            return []
        if doc_key not in self._doc_vectors:
            self._doc_vectors[doc_key] = self.embeddings.embed_documents([d.page_content for d in pool])
        query_vector = self.embeddings.embed_query(query)
        similarity = [sum(a * b for a, b in zip(query_vector, v)) for v in self._doc_vectors[doc_key]]
        semantic = [pool[i] for i in sorted(range(len(pool)), key=similarity.__getitem__, reverse=True)]
        keyword = BM25Okapi([_tokens(d.page_content) for d in pool]).get_top_n(_tokens(query), pool, n=len(pool))
        scores: defaultdict[str, float] = defaultdict(float)
        for ranked in (semantic, keyword):
            for rank, document in enumerate(ranked, start=1):
                scores[document.metadata["source_id"]] += 1.0 / (60 + rank)
        best = sorted(scores, key=scores.__getitem__, reverse=True)[:top_k]
        return [self.documents_by_id[source_id] for source_id in best]

    def search(self, query: str, top_k: int = 5, domain: str | None = None) -> list[Document]:
        """Domain-first: top results from the chosen domain, plus a global backstop
        (GLOBAL_BACKSTOP best chunks from all domains) in case triage picked the wrong domain."""
        if not query.strip():
            return []

        if domain not in self.pools or domain == "all":
            ids = self._ranked(query, "all", top_k)[:top_k]
        else:
            in_domain = self._ranked(query, domain, top_k)
            global_ids = [i for i in self._ranked(query, "all", top_k) if i not in in_domain]
            ids = in_domain[: top_k - GLOBAL_BACKSTOP] + global_ids[:GLOBAL_BACKSTOP]
            ids += [i for i in in_domain if i not in ids][: top_k - len(ids)]
        return [self.documents_by_id[source_id] for source_id in ids]

    def add_context(self, query: str, evidence: list[Document], domain: str) -> list[Document]:
        """Add what the retrieved chunks need to be complete: the table of each retrieved finance
        report, and the best keyword matches for error codes / identifiers in the question."""
        out = list(evidence)
        added = 0
        for document in evidence:
            if document.metadata["domain"] != "finance":
                continue
            config, row, _ = document.metadata["doc_key"].rsplit(":", 2)
            for key in ROW_DOCS.get(f"{config}:{row}", []):
                chunks = self.by_doc.get(key, [])
                if not chunks or not chunks[0].page_content.lstrip().startswith("[["):
                    continue  # FinQA tables are stored as '[["", "2014", ...' rows
                for chunk in chunks:
                    if chunk not in out and added < REPORT_TABLES:
                        out.append(chunk)
                        added += 1
        codes = [
            token for token in _tokens(query)
            if re.search(r"\d", token) and len(token) >= 4 and not re.fullmatch(r"(19|20)\d\d", token)
        ]
        if codes and domain in self.pools and domain not in ("legal", "all"):
            pool = self.pools[domain]
            scores = self.bm25[domain].get_scores(codes)
            best = sorted(range(len(pool)), key=scores.__getitem__, reverse=True)
            out += [pool[i] for i in best[: CODE_EXTRA + len(out)] if scores[i] > 0 and pool[i] not in out][:CODE_EXTRA]
        return out

_NAME_STOPWORDS = {
    "the", "a", "an", "of", "and", "or", "in", "on", "to", "for", "by", "between", "among", "with", "this",
    "agreement", "contract", "inc", "llc", "ltd", "lp", "corp", "corporation", "company", "co", "limited",
    "holdings", "group", "dated", "as",
}


def match_contract(name: str) -> str | None:
    """Return the doc_key of the contract whose title/parties best match `name`, if the match is clear."""
    wanted = {t for t in _tokens(name) if t not in _NAME_STOPWORDS and len(t) > 2}
    if not wanted:
        return None
    scored = []
    for doc_key, head in CONTRACT_HEADS.items():
        have = set(_tokens(head[:600]))
        scored.append((len(wanted & have) / len(wanted), doc_key))
    scored.sort(reverse=True)
    best_score, best_key = scored[0]
    runner_up = scored[1][0] if len(scored) > 1 else 0.0
    return best_key if best_score >= 0.6 and best_score > runner_up else None


## 7. LLM helper and OpenFDA drug-label lookup

In [ ]:
def _invoke_structured(
    llm: ChatGroq,
    schema: type[BaseModel],
    messages: list[tuple[str, str]],
) -> BaseModel:
    # json_schema mode avoids gpt-oss mis-casing the tool name (e.g. "triageResult").
    # gpt-oss occasionally echoes the schema or returns invalid JSON; retry before failing.
    for attempt in range(3):
        try:
            result = llm.with_structured_output(schema, method="json_schema").invoke(messages)
            if isinstance(result, dict):
                result = schema.model_validate(result)
            if isinstance(result, schema):
                return result
        except Exception as error:
            if "rate_limit" in str(error) or attempt == 2:
                raise
    raise TypeError(f"Expected {schema.__name__} from structured model output.")


_DASH = "[-\u2010-\u2015\u2212]"
_CITE_GROUP = re.compile(
    rf"[\[\u3010(]\s*(?:(?:RAG|FDA){_DASH}\d+\s*(?:[,;/&]|and)?\s*)+[\]\u3011)]"
)
_CITE_BARE = re.compile(rf"(?<![\[\w])(RAG|FDA){_DASH}(\d+)(?![\]\w])")


def _normalize_citations(text: str) -> str:
    """Rewrite every way gpt-oss writes a source ID as [RAG-00001]: full-width or round
    brackets, Unicode hyphens, grouped IDs ([RAG-1, RAG-2]) and bare/bold IDs (**RAG-1**)."""
    text = _CITE_GROUP.sub(
        lambda m: "".join(
            f"[{prefix}-{number}]"
            for prefix, number in re.findall(rf"(RAG|FDA){_DASH}(\d+)", m.group(0))
        ),
        text,
    )
    return _CITE_BARE.sub(r"[\1-\2]", text)


def _openfda_query(medication: str) -> tuple[dict[str, str], str]:
    escaped = medication.replace("\\", "\\\\").replace('"', '\\"')
    search = f'openfda.generic_name:"{escaped}" OR openfda.brand_name:"{escaped}"'
    params = {"search": search, "limit": "1"}
    prepared = requests.Request("GET", OPENFDA_LABEL_URL, params=params).prepare()
    if prepared.url is None:
        raise RuntimeError("Could not construct the OpenFDA query URL.")
    return params, prepared.url


def _fetch_drug_labels(medications: list[str]) -> tuple[list[DrugLabel], list[str]]:
    labels: list[DrugLabel] = []
    notes: list[str] = []

    for index, medication in enumerate(medications[:MAX_MEDICATIONS], start=1):
        params, url = _openfda_query(medication)
        source_id = f"FDA-{index:03d}"
        try:
            response = requests.get(OPENFDA_LABEL_URL, params=params, timeout=15)
            if response.status_code == 404:
                labels.append({"source_id": source_id, "medication": medication,
                               "status": "not found", "url": response.url, "text": ""})
                notes.append(
                    f"{source_id}: OpenFDA returned no matching label for "
                    f"{medication!r}; this does not establish safety."
                )
                continue
            response.raise_for_status()
            payload = response.json()
        except requests.RequestException as error:
            labels.append({"source_id": source_id, "medication": medication,
                           "status": "unavailable", "url": url, "text": ""})
            notes.append(f"{source_id}: OpenFDA lookup failed for {medication!r}: {error}")
            continue

        results = payload.get("results", [])
        if not results:
            labels.append({"source_id": source_id, "medication": medication,
                           "status": "not found", "url": response.url, "text": ""})
            notes.append(
                f"{source_id}: no matching OpenFDA label for {medication!r}; "
                "this does not establish safety."
            )
            continue

        result = results[0]
        sections: list[str] = []
        for section_name in ("drug_interactions", "contraindications", "warnings"):
            values = result.get(section_name, [])
            if isinstance(values, str):
                values = [values]
            for value in values:
                if isinstance(value, str) and value.strip():
                    sections.append(f"{section_name}: {value.strip()}")

        text = "\n\n".join(sections)[:MAX_LABEL_CHARS]
        if not text:
            notes.append(
                f"{source_id}: a label was found for {medication!r}, but the "
                "requested interaction/warning sections were absent."
            )
        labels.append({"source_id": source_id, "medication": medication,
                       "status": "retrieved", "url": response.url, "text": text})

    if len(medications) > MAX_MEDICATIONS:
        notes.append(f"Only the first {MAX_MEDICATIONS} medication names were checked.")
    return labels, notes

## 8. LangGraph workflow (the 5 agents)

| Node | Groq call? | Output |
|---|---|---|
| `triage` | yes (`TriageResult`) | severity, emergency flag, medications |
| `hybrid_retrieval` | no | top-8 evidence chunks (`TOP_K`), plus finance tables and error-code matches |
| `drug_interaction_check` | only when medications are given | OpenFDA labels + verified findings |
| `evidence_synthesis` | yes (free text) | cited draft answer |
| `safety_gate` | yes (`SafetyAssessment`) | final answer, review flag, notes |

In [ ]:
from collections import Counter


def create_workflow(retriever: HybridRetriever, llm: ChatGroq):
    def triage_node(state: ClinicalState) -> dict[str, object]:
        triage = _invoke_structured(
            llm,
            TriageResult,
            [
                (
                    "system",
                    "Triage a question for a research-only multi-domain evidence "
                    "assistant and classify its knowledge domain. Do not diagnose. Mark emergencies and "
                    "individualized medication/treatment decisions for clinician "
                    "review. Extract only medication names explicitly present. If the question "
                    "names a specific contract, agreement, company or parties, copy that name into "
                    "document_name.",
                ),
                (
                    "human",
                    f"Question: {state['question']}\n"
                    f"User-supplied medication list: "
                    f"{', '.join(state.get('user_medications', [])) or 'none'}",
                ),
            ],
        )
        if not isinstance(triage, TriageResult):
            raise TypeError("Triage agent returned an invalid result.")

        medications = list(
            dict.fromkeys([*state.get("user_medications", []), *triage.medications])
        )
        safety_notes = list(state.get("safety_notes", []))
        if len(medications) > MAX_MEDICATIONS:
            safety_notes.append(
                f"Only the first {MAX_MEDICATIONS} medication names were screened."
            )
        return {
            "triage": triage,
            "medications": medications[:MAX_MEDICATIONS],
            "safety_notes": safety_notes,
        }

    def retrieval_node(state: ClinicalState) -> dict[str, object]:
        triage = state["triage"]
        domain = triage.domain
        notes = list(state.get("safety_notes", []))
        # Contract-scoped search: a contract picked by the user, or one named in the question.
        doc_key = state.get("document") or (
            match_contract(triage.document_name) if triage.document_name else None
        )
        if doc_key in CONTRACTS:
            notes.append(f"Searched only the contract: {CONTRACTS[doc_key]}")
            return {
                "triage": triage.model_copy(update={"domain": "legal"}),
                "safety_notes": notes,
                "evidence": retriever.search_document(state["question"], doc_key, top_k=TOP_K),
            }
        if domain == "legal" and len(CONTRACTS) > 1:
            notes.append(
                "No contract was named, so the answer may come from any of the "
                f"{len(CONTRACTS)} indexed contracts; name the contract or pick it in the app."
            )
        # Check triage against the evidence: if most of the best chunks across all domains
        # come from one other domain, the question belongs there.
        votes = Counter(
            document.metadata["domain"]
            for document in retriever.search(state["question"], top_k=5, domain=None)
        )
        if votes:
            best, count = votes.most_common(1)[0]
            if best != domain and count >= DOMAIN_VOTE_MIN and votes[domain] <= 1:
                notes.append(f"Domain changed from {domain} to {best} by the retrieval check.")
                domain = best
                triage = triage.model_copy(update={"domain": best})
        return {
            "triage": triage,
            "safety_notes": notes,
            "evidence": retriever.add_context(
                state["question"],
                retriever.search(state["question"], top_k=TOP_K, domain=None if domain == "unknown" else domain),
                domain,
            ),
        }

    def medication_check_node(state: ClinicalState) -> dict[str, object]:
        labels, notes = _fetch_drug_labels(state.get("medications", []))
        notes = [*state.get("safety_notes", []), *notes]
        available_labels = [
            label for label in labels if label["status"] == "retrieved" and label["text"]
        ]
        if available_labels:
            label_context = "\n\n".join(
                f"[{label['source_id']}] Medication: {label['medication']}\n{label['text']}"
                for label in available_labels
            )
            assessment = _invoke_structured(
                llm,
                InteractionAssessment,
                [
                    (
                        "system",
                        "Screen medication names only against the supplied OpenFDA "
                        "label excerpts. Report a possible interaction only when "
                        "an exact excerpt explicitly supports it. Never infer that "
                        "a combination is safe. A label excerpt may not cover all "
                        "interactions. Use only supplied FDA source IDs and quotes.",
                    ),
                    (
                        "human",
                        f"Medication names to screen: {state.get('medications', [])}\n"
                        f"Label excerpts:\n{label_context}",
                    ),
                ],
            )
            if not isinstance(assessment, InteractionAssessment):
                raise TypeError("Medication interaction agent returned an invalid result.")

            verified_findings: list[InteractionFinding] = []
            label_by_id = {label["source_id"]: label for label in available_labels}
            for finding in assessment.findings:
                source = label_by_id.get(finding.source_id)
                if (
                    source is not None
                    and finding.exact_quote.strip()
                    and finding.exact_quote.strip() in source["text"]
                ):
                    verified_findings.append(finding)
                else:
                    notes.append(
                        "An interaction statement from the model was omitted "
                        "because its source quote could not be verified."
                    )
            assessment = assessment.model_copy(
                update={
                    "findings": verified_findings,
                    "summary": (
                        "Potential interaction information was found in label "
                        "text; have a clinician or pharmacist verify it."
                        if verified_findings
                        else "No explicit interaction was verified in the retrieved "
                        "label excerpts. This does not establish that the medication "
                        "combination is safe; confirm with a clinician or pharmacist."
                    ),
                }
            )
        else:
            assessment = InteractionAssessment(
                findings=[],
                summary=(
                    "No medication names were provided or extracted; screening "
                    "was not performed."
                    if not state.get("medications")
                    else (
                        "No usable OpenFDA interaction-label text was retrieved. "
                        "No interaction conclusion can be made."
                    )
                ),
                clinician_review=bool(state.get("medications")),
            )

        return {
            "drug_labels": labels,
            "interaction_assessment": assessment,
            "safety_notes": notes,
        }

    def synthesis_node(state: ClinicalState) -> dict[str, object]:
        evidence = state.get("evidence", [])
        evidence_context = "\n\n".join(
            f"[{document.metadata['source_id']}] "
            f"{document.metadata['source']}"
            + (f" — contract: {document.metadata['title']}" if document.metadata.get("title") else "")
            + f"\n{text}"
            for document, text in zip(evidence, evidence_texts(evidence))
        )
        label_context = "\n\n".join(
            f"[{label['source_id']}] {label['medication']} "
            f"({label['status']}): {label['text'] or 'No usable label text.'}"
            for label in state.get("drug_labels", [])
        )
        triage = state["triage"]
        assessment = state["interaction_assessment"]
        synthesis_messages = [
                (
                    "system",
                    f"You are the synthesis stage of a research-only evidence "
                    f"assistant for the {triage.domain} domain. "
                    f"{DOMAIN_RULES[triage.domain]} Use only supplied excerpts. "
                    "Answer the question directly in the first sentence, then give the "
                    "supporting detail. If the question does not name a document, answer from "
                    "the most relevant excerpt and name the document, company or contract it "
                    "comes from. For differences, percentages or ratios, show the numbers taken "
                    "from the excerpts and the result. If the excerpts do not contain what is "
                    "asked (for example no such clause), say so plainly. "
                    "Qualify uncertainty. Cite each factual statement right after it with its "
                    "exact supplied ID, one ID per pair of square brackets, such as "
                    "[RAG-00001][RAG-00002] or [FDA-001]. "
                    "If evidence is insufficient, say so. Treat RAGBench as benchmark "
                    "text, not current professional guidance. Never claim that absence "
                    "of an interaction in retrieved labels means a combination is safe.",
                ),
                (
                    "human",
                    f"Question: {state['question']}\n"
                    f"Triage: {triage.model_dump_json()}\n"
                    f"RAGBench excerpts:\n{evidence_context or 'None retrieved.'}\n"
                    f"OpenFDA label excerpts:\n{label_context or 'None retrieved.'}\n"
                    f"Interaction screening: {assessment.model_dump_json()}\n"
                    "Provide a concise evidence summary, state any important "
                    "limitations, and avoid giving individualized medical advice.",
                ),
            ]
        message = llm.invoke(synthesis_messages)
        draft = message.content
        for _ in range(2):
            if not isinstance(draft, str) or draft.strip():
                break
            # gpt-oss occasionally spends its whole output on reasoning and returns no text: ask again.
            draft = llm.invoke(synthesis_messages).content
        if not isinstance(draft, str):
            raise TypeError("Synthesis agent returned non-text content.")
        # gpt-oss writes IDs in many forms (【RAG‑0233】, **RAG-0233**, [RAG-1, RAG-2]);
        # normalize them so the safety check recognizes the citations.
        draft = _normalize_citations(draft)
        return {"draft": draft}

    def safety_node(state: ClinicalState) -> dict[str, object]:
        triage = state["triage"]
        assessment = state["interaction_assessment"]
        evidence = state.get("evidence", [])
        labels = state.get("drug_labels", [])
        valid_ids = {document.metadata["source_id"] for document in evidence} | {
            label["source_id"]
            for label in labels
            if label["status"] == "retrieved" and label["text"]
        }
        allowed_ids = {f"[{source_id}]" for source_id in valid_ids}
        notes = list(state.get("safety_notes", []))
        draft = state["draft"]
        unknown_ids = set(re.findall(r"\[(?:RAG|FDA)-\d+\]", draft)) - allowed_ids
        if unknown_ids:
            # Drop citations to sources that were not retrieved instead of discarding the whole answer;
            # the statements stay and are still checked by the auditor below.
            for source_id in unknown_ids:
                draft = draft.replace(source_id, "")
            notes.append("Citations to sources that were not retrieved were removed: " + ", ".join(sorted(unknown_ids)))
        used_ids = set(re.findall(r"\[(?:RAG|FDA)-\d+\]", draft))
        unsafe = False
        deterministic_emergency = any(
            re.search(pattern, state["question"], flags=re.IGNORECASE)
            for pattern in EMERGENCY_PATTERNS
        )
        if deterministic_emergency and not triage.emergency_concern:
            notes.append("A possible emergency symptom matched a conservative safety rule.")

        if not evidence:
            unsafe = True
            notes.append("No RAGBench evidence was retrieved for this question.")
        if not state["draft"].strip():
            unsafe = True
            notes.append("The synthesis step returned an empty answer.")
        uncited = bool(evidence and not used_ids)
        if uncited:
            notes.append("The answer is not tied to specific source IDs; check it against the sources.")

        safety_review = _invoke_structured(
            llm,
            SafetyAssessment,
            [
                (
                    "system",
                    "Audit the draft for claims not supported by the evidence and for "
                    f"breaches of this rule: {DOMAIN_RULES[triage.domain]} "
                    "A claim directly supported by a cited excerpt counts as supported, "
                    "even if only one excerpt supports it. A statement that the excerpts do not "
                    "contain something (for example 'the contract has no such clause') is supported "
                    "when the evidence indeed does not contain it. Judge each claim against all of "
                    "the evidence, not only the excerpt it cites. Do not rewrite the draft. "
                    "Flag uncertainty for expert review.",
                ),
                (
                    "human",
                    f"Question: {state['question']}\n"
                    f"Evidence:\n"
                    + "\n\n".join(
                        f"[{document.metadata['source_id']}] {text}"
                        for document, text in zip(evidence, evidence_texts(evidence))
                    )
                    + "\n\nDraft:\n"
                    + draft,
                ),
            ],
        )
        if not isinstance(safety_review, SafetyAssessment):
            raise TypeError("Safety agent returned an invalid result.")
        answer = draft
        claims = safety_review.unsafe_or_unsupported_claims
        if claims and not unsafe:
            # Repair: drop only the flagged statements instead of discarding the whole answer.
            revision = llm.invoke(
                [
                    (
                        "system",
                        "Remove every listed statement from the draft, plus anything that "
                        "depends on it. Keep all other text and its square-bracket source IDs "
                        "unchanged. Do not add new content. Return only the revised draft.",
                    ),
                    (
                        "human",
                        "Statements to remove:\n- " + "\n- ".join(claims)
                        + "\n\nDraft:\n" + draft,
                    ),
                ]
            ).content
            revised = _normalize_citations(revision) if isinstance(revision, str) else ""
            for source_id in set(re.findall(r"\[(?:RAG|FDA)-\d+\]", revised)) - allowed_ids:
                revised = revised.replace(source_id, "")
            if revised.strip():
                answer = revised
                notes.append("Statements the evidence did not support were removed: " + "; ".join(claims))
            else:
                unsafe = True
                notes.extend(claims)

        review_required = (
            triage.severity in {"high", "critical"}
            or triage.emergency_concern
            or deterministic_emergency
            or triage.clinician_review
            or assessment.clinician_review
            or bool(assessment.findings)
            or safety_review.requires_clinician_review
            or bool(state.get("medications"))
            or uncited
            or bool(unknown_ids)
            or bool(claims)
        )

        if unsafe:
            answer = (
                "I can't provide a dependable summary from the retrieved "
                "evidence. Please have a qualified expert review "
                "the question and the cited source material."
            )
            review_required = True

        if triage.emergency_concern or triage.severity == "critical" or deterministic_emergency:
            answer = (
                "This may require urgent in-person assessment. Contact your local "
                "emergency number or seek emergency care now. Do not rely on this "
                "research prototype for emergency decisions."
            )
            review_required = True

        return {"answer": answer, "review_required": review_required, "safety_notes": notes}

    workflow = StateGraph(ClinicalState)
    workflow.add_node("triage", triage_node)
    workflow.add_node("hybrid_retrieval", retrieval_node)
    workflow.add_node("drug_interaction_check", medication_check_node)
    workflow.add_node("evidence_synthesis", synthesis_node)
    workflow.add_node("safety_gate", safety_node)
    workflow.set_entry_point("triage")
    workflow.add_edge("triage", "hybrid_retrieval")
    workflow.add_edge("hybrid_retrieval", "drug_interaction_check")
    workflow.add_edge("drug_interaction_check", "evidence_synthesis")
    workflow.add_edge("evidence_synthesis", "safety_gate")
    workflow.add_edge("safety_gate", END)
    return workflow.compile()

## 9. Build the assistant (loads data, embeds chunks, compiles the graph)

In [ ]:
llm = ChatGroq(model=GROQ_MODEL, temperature=0, max_retries=2, timeout=60)
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": EMBEDDING_DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 256 if EMBEDDING_DEVICE == "cuda" else 32},
)
documents = load_ragbench_documents()
print(f"Loaded {len(documents)} chunks from {len(DOMAINS)} domains.")
assistant = create_workflow(HybridRetriever(documents, embeddings), llm)
print("Assistant ready.")

## 10. Format results

In [ ]:
def format_markdown(state: ClinicalState) -> str:
    triage = state["triage"]
    assessment = state["interaction_assessment"]
    lines = []
    if state["review_required"]:
        lines.append("**⚠️ EXPERT REVIEW RECOMMENDED**\n")
    lines += [
        state["answer"],
        "",
        "---",
        f"**Triage:** {triage.domain} · {triage.severity} · {triage.category} — {triage.rationale}",
        "",
        f"**Medication-label screen:** {assessment.summary}",
    ]
    for finding in assessment.findings:
        lines.append(
            f"- Potential interaction ({', '.join(finding.medications)}): "
            f"\"{finding.exact_quote}\" [{finding.source_id}]"
        )
    if state.get("evidence"):
        lines += ["", f"**Retrieved evidence · {len(state['evidence'])} sources**"]
        for document in state["evidence"]:
            lines.append(
                f"- [{document.metadata['source_id']}] {document.metadata['source']}"
                + (f" — *{document.metadata['title']}*" if document.metadata.get("title") else "")
            )
    if state.get("drug_labels"):
        lines += ["", "**OpenFDA label lookups**"]
        for label in state["drug_labels"]:
            lines.append(f"- [{label['source_id']}] {label['medication']}: {label['status']}")
    if state.get("safety_notes"):
        lines += ["", "**Safety and availability notes**"]
        lines += [f"- {note}" for note in state["safety_notes"]]
    lines += ["", f"*{DISCLAIMER}*"]
    return "\n".join(lines)


def ask(question: str, medications: list[str] | None = None, document: str = "") -> ClinicalState:
    """Run one question through the pipeline and display the result.
    `document` (optional): a contract doc_key from CONTRACTS to search only that contract."""
    state = assistant.invoke(
        {"question": question.strip(), "user_medications": medications or [], "document": document}
    )
    try:
        from IPython.display import Markdown, display
        display(Markdown(format_markdown(state)))
    except ImportError:
        print(format_markdown(state))
    return state

## 11. Try it directly

One question per domain, each taken from the indexed rows:
biomedical *What is the incubation period of MERS?* ·
finance *What is the yearly amortization rate related to the trademarks?* ·
customer support *Why does the other instance of my multi-instance qmgr seem to hang after a failover? Queue manager will not start after failover.* ·
general *In what school district is Governor John R. Rogers High School, named after John Rankin Rogers, located?*

In [ ]:
result = ask("What is the incubation period of MERS?")

# Legal: name the contract in the question, or pass it explicitly (see CONTRACTS for all titles).
first_contract = next(iter(CONTRACTS))
print("Example contract:", CONTRACTS[first_contract])
result = ask("Is one party required to deposit its source code into escrow?", document=first_contract)


## 12. Chat UI (Gradio)

Launches with share=True: the output shows a temporary public https://....gradio.live link
(live up to 72 hours while this cell runs; anyone with the link can use your Groq key).

In [ ]:
import gradio as gr


ANY_CONTRACT = "(any contract — or name it in the question)"
CONTRACT_CHOICES = [ANY_CONTRACT] + [f"{title}  [{key}]" for key, title in CONTRACTS.items()]


def chat(message, history, medications_text, contract_choice):
    medications = [name.strip() for name in (medications_text or "").split(",") if name.strip()]
    document = ""
    if contract_choice and contract_choice != ANY_CONTRACT:
        document = contract_choice.rsplit("[", 1)[1].rstrip("]")
    try:
        state = assistant.invoke(
            {"question": message.strip(), "user_medications": medications, "document": document}
        )
    except Exception as error:
        return f"The assistant could not complete the request: {error}"
    return format_markdown(state)


demo = gr.ChatInterface(
    fn=chat,
    title="Multi-Domain RAG Agents — research prototype",
    description="Evidence from 5 RAGBench domains (biomedical, legal, finance, support, general), answered by Groq.",
    additional_inputs=[
        gr.Textbox(label="Medications to screen (comma-separated, optional)"),
        gr.Dropdown(CONTRACT_CHOICES, value=ANY_CONTRACT, label="Contract (legal questions, optional)"),
    ],
    examples=[
        ["What is the incubation period of MERS?", "", ANY_CONTRACT],
        ["What is the yearly amortization rate related to the trademarks?", "", ANY_CONTRACT],
        ["Why does the other instance of my multi-instance qmgr seem to hang after a failover? Queue manager will not start after failover.", "", ANY_CONTRACT],
        ["In what school district is Governor John R. Rogers High School, named after John Rankin Rogers, located?", "", ANY_CONTRACT],
    ],
)
# debug=False: the cell finishes once the public link is printed; the app keeps running
# in the background until the runtime stops (debug=True would keep this cell "running" forever).
demo.launch(share=True, debug=False)